In [13]:
import pandas as pd

# Upload your files 
files = [
    'C:\\Users\\Riyad\\projects\\medical-assistand-ml\\Original_Dataset.csv',
    'C:\\Users\\Riyad\\projects\\medical-assistand-ml\\Symptom_Weights.csv',
    'C:\\Users\\Riyad\\projects\\medical-assistand-ml\\Disease_Description.csv',
    'C:\\Users\\Riyad\\projects\\medical-assistand-ml\\medicine.csv',
    'C:\\Users\\Riyad\\projects\\medical-assistand-ml\\Doctor_Specialist.csv',
    'C:\\Users\\Riyad\\projects\\medical-assistand-ml\\Doctor_Versus_Disease.csv'
]

for file in files:
    try:
        df = pd.read_csv(file)
        print(f"\n{'='*60}")
        print(f" File: {file}")
        print(f"{'='*60}")
        print(f"Shape: {df.shape}")
        print(f"Columns: {list(df.columns)}")
        print(f"\nFirst 3 rows:")
        print(df.head(3))
    except Exception as e:
        print(f"Error reading {file}: {e}")


 File: C:\Users\Riyad\projects\medical-assistand-ml\Original_Dataset.csv
Shape: (4920, 18)
Columns: ['Disease', 'Symptom_1', 'Symptom_2', 'Symptom_3', 'Symptom_4', 'Symptom_5', 'Symptom_6', 'Symptom_7', 'Symptom_8', 'Symptom_9', 'Symptom_10', 'Symptom_11', 'Symptom_12', 'Symptom_13', 'Symptom_14', 'Symptom_15', 'Symptom_16', 'Symptom_17']

First 3 rows:
            Disease   Symptom_1              Symptom_2              Symptom_3  \
0  Fungal infection     itching              skin_rash   nodal_skin_eruptions   
1  Fungal infection   skin_rash   nodal_skin_eruptions    dischromic _patches   
2  Fungal infection     itching   nodal_skin_eruptions    dischromic _patches   

              Symptom_4 Symptom_5 Symptom_6 Symptom_7 Symptom_8 Symptom_9  \
0   dischromic _patches       NaN       NaN       NaN       NaN       NaN   
1                   NaN       NaN       NaN       NaN       NaN       NaN   
2                   NaN       NaN       NaN       NaN       NaN       NaN   

  Symptom

In [3]:
!pip install scikit-learn pandas numpy joblib -q


[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import joblib
import warnings
warnings.filterwarnings('ignore')

print(" Libraries imported successfully!\n")

 Libraries imported successfully!



In [16]:
df_original = pd.read_csv('C:\\Users\\Riyad\\projects\\medical-assistand-ml\\Original_Dataset.csv')
df_symptom_weights = pd.read_csv('C:\\Users\\Riyad\\projects\\medical-assistand-ml\\Symptom_Weights.csv')
df_disease_desc = pd.read_csv( 'C:\\Users\\Riyad\\projects\\medical-assistand-ml\\Disease_Description.csv')
df_medicine = pd.read_csv('C:\\Users\\Riyad\\projects\\medical-assistand-ml\\medicine.csv')
df_doctor_vs_disease = pd.read_csv(
    'C:\\Users\\Riyad\\projects\\medical-assistand-ml\\Doctor_Versus_Disease.csv',
    encoding='latin-1'  
)

print(df_doctor_vs_disease.head())
print(df_doctor_vs_disease.columns.tolist())
print(f" Original Dataset: {df_original.shape}")
print(f" Symptom Weights: {df_symptom_weights.shape}")
print(f" Disease Description: {df_disease_desc.shape}")
print(f" Medicine Dataset: {df_medicine.shape}\n")

   Drug Reaction      Allergist
0        Allergy      Allergist
1  Hypertension    Cardiologist
2   Heart attack   Cardiologist
3      Psoriasis  Dermatologist
4    Chicken pox  Dermatologist
['Drug Reaction', 'Allergist']
 Original Dataset: (4920, 18)
 Symptom Weights: (130, 2)
 Disease Description: (41, 2)
 Medicine Dataset: (21714, 10)



In [17]:
print(" Preprocessing data...")

# Get all unique symptoms from the dataset
symptom_columns = [col for col in df_original.columns if col.startswith('Symptom_')]
print(f" Found {len(symptom_columns)} symptom columns")

# Get all unique symptoms (excluding NaN)
all_symptoms = set()
for col in symptom_columns:
    symptoms = df_original[col].dropna().unique()
    all_symptoms.update(symptoms)

all_symptoms = sorted(list(all_symptoms))
print(f" Total unique symptoms: {len(all_symptoms)}\n")

# Create symptom to index mapping
symptom_to_index = {symptom: idx for idx, symptom in enumerate(all_symptoms)}

 Preprocessing data...
 Found 17 symptom columns
 Total unique symptoms: 131



In [18]:
print(" Creating feature matrix...")

def create_feature_vector(row, symptom_columns, symptom_to_index):
    """Convert symptoms to binary feature vector"""
    feature_vector = np.zeros(len(symptom_to_index))

    for col in symptom_columns:
        symptom = row[col]
        if pd.notna(symptom) and symptom in symptom_to_index:
            feature_vector[symptom_to_index[symptom]] = 1

    return feature_vector

# Create features (X) and labels (y)
X = []
y = []

for idx, row in df_original.iterrows():
    feature_vec = create_feature_vector(row, symptom_columns, symptom_to_index)
    X.append(feature_vec)
    y.append(row['Disease'])

    if (idx + 1) % 1000 == 0:
        print(f"Processed {idx + 1}/{len(df_original)} rows...")

X = np.array(X)
y = np.array(y)

print(f"\n Feature matrix shape: {X.shape}")
print(f" Labels shape: {y.shape}")
print(f" Number of unique diseases: {len(np.unique(y))}\n")

 Creating feature matrix...
Processed 1000/4920 rows...
Processed 2000/4920 rows...
Processed 3000/4920 rows...
Processed 4000/4920 rows...

 Feature matrix shape: (4920, 131)
 Labels shape: (4920,)
 Number of unique diseases: 41



In [19]:
print(" Encoding disease labels...")

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

print(f" Encoded labels shape: {y_encoded.shape}\n")

 Encoding disease labels...
 Encoded labels shape: (4920,)



In [20]:
print(" Splitting dataset into train and test...")

X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

print(f" Training set: {X_train.shape}")
print(f" Test set: {X_test.shape}\n")

 Splitting dataset into train and test...
 Training set: (3936, 131)
 Test set: (984, 131)



In [21]:
print(" Training Random Forest model...")


model = RandomForestClassifier(
    n_estimators=100,
    max_depth=20,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

model.fit(X_train, y_train)

print("\n Model training completed!\n")

 Training Random Forest model...

 Model training completed!



[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=-1)]: Done  26 tasks      | elapsed:    0.0s
[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:    0.1s finished


In [22]:
print("Evaluating model...")

y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"\n Model Accuracy: {accuracy * 100:.2f}%\n")

print(" Classification Report:\n")
print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))

Evaluating model...

 Model Accuracy: 100.00%

 Classification Report:

                                         precision    recall  f1-score   support

(vertigo) Paroymsal  Positional Vertigo       1.00      1.00      1.00        24
                                   AIDS       1.00      1.00      1.00        24
                                   Acne       1.00      1.00      1.00        24
                    Alcoholic hepatitis       1.00      1.00      1.00        24
                                Allergy       1.00      1.00      1.00        24
                              Arthritis       1.00      1.00      1.00        24
                       Bronchial Asthma       1.00      1.00      1.00        24
                   Cervical spondylosis       1.00      1.00      1.00        24
                            Chicken pox       1.00      1.00      1.00        24
                    Chronic cholestasis       1.00      1.00      1.00        24
                            Common C

[Parallel(n_jobs=12)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done  26 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 100 out of 100 | elapsed:    0.0s finished


In [23]:
print("\n Saving model and supporting files...")

# Save model
joblib.dump(model, 'disease_prediction_model.pkl')
print(" Saved: disease_prediction_model.pkl")

# Save label encoder
joblib.dump(label_encoder, 'label_encoder.pkl')
print(" Saved: label_encoder.pkl")

# Save symptom to index mapping
joblib.dump(symptom_to_index, 'symptom_to_index.pkl')
print(" Saved: symptom_to_index.pkl")

# Save all symptoms list
joblib.dump(all_symptoms, 'all_symptoms.pkl')
print(" Saved: all_symptoms.pkl")

# Save disease descriptions
joblib.dump(df_disease_desc, 'disease_descriptions.pkl')
print(" Saved: disease_descriptions.pkl")

# Save medicine data (simplified)
medicine_by_indication = df_medicine.groupby('type').agg({
    'brand name': lambda x: list(x)[:5]  # Top 5 medicines per type
}).to_dict()
joblib.dump(medicine_by_indication, 'medicine_data.pkl')
print(" Saved: medicine_data.pkl")

print("\n" + "="*60)
print(" ALL FILES SAVED SUCCESSFULLY!")
print("="*60)


 Saving model and supporting files...
 Saved: disease_prediction_model.pkl
 Saved: label_encoder.pkl
 Saved: symptom_to_index.pkl
 Saved: all_symptoms.pkl
 Saved: disease_descriptions.pkl
 Saved: medicine_data.pkl

 ALL FILES SAVED SUCCESSFULLY!


In [24]:
print("\n Testing prediction with sample symptoms...")

def predict_disease(symptoms_list, model, label_encoder, symptom_to_index):
    """Predict disease from list of symptoms"""
    feature_vector = np.zeros(len(symptom_to_index))

    for symptom in symptoms_list:
        if symptom in symptom_to_index:
            feature_vector[symptom_to_index[symptom]] = 1

    feature_vector = feature_vector.reshape(1, -1)
    prediction = model.predict(feature_vector)
    disease = label_encoder.inverse_transform(prediction)[0]

    # Get probability
    probabilities = model.predict_proba(feature_vector)[0]
    confidence = max(probabilities) * 100

    return disease, confidence

# Test with sample symptoms
test_symptoms = ['itching', 'skin_rash', 'nodal_skin_eruptions']
predicted_disease, confidence = predict_disease(
    test_symptoms, model, label_encoder, symptom_to_index
)

print(f"\n Input Symptoms: {test_symptoms}")
print(f" Predicted Disease: {predicted_disease}")
print(f" Confidence: {confidence:.2f}%")

# Get description
if predicted_disease in df_disease_desc['Disease'].values:
    desc = df_disease_desc[df_disease_desc['Disease'] == predicted_disease]['Description'].values[0]
    print(f"Description: {desc[:500]}...")

print("\n" + "="*60)
print("MODEL TRAINING COMPLETE!")
print("="*60)
print("\n Download these files for API:")
print("   - disease_prediction_model.pkl")
print("   - label_encoder.pkl")
print("   - symptom_to_index.pkl")
print("   - all_symptoms.pkl")
print("   - disease_descriptions.pkl")
print("   - medicine_data.pkl")
print("\n Next step: Create Flask/FastAPI for deployment!")


 Testing prediction with sample symptoms...

 Input Symptoms: ['itching', 'skin_rash', 'nodal_skin_eruptions']
 Predicted Disease: Drug Reaction
 Confidence: 7.28%
Description: An adverse drug reaction (ADR) is an injury caused by taking medication. ADRs may occur following a single dose or prolonged administration of a drug or result from the combination of two or more drugs....

MODEL TRAINING COMPLETE!

 Download these files for API:
   - disease_prediction_model.pkl
   - label_encoder.pkl
   - symptom_to_index.pkl
   - all_symptoms.pkl
   - disease_descriptions.pkl
   - medicine_data.pkl

 Next step: Create Flask/FastAPI for deployment!


[Parallel(n_jobs=12)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done  26 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 100 out of 100 | elapsed:    0.0s finished
[Parallel(n_jobs=12)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done  26 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 100 out of 100 | elapsed:    0.0s finished


In [25]:
!pip install fastapi uvicorn pyngrok python-multipart -q
!pip install rapidfuzz nltk scikit-learn joblib -q
!pip install nest-asyncio -q


[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


     ---------------------------------------- 0.0/627.2 kB ? eta -:--:--
     - ------------------------------------- 30.7/627.2 kB 1.3 MB/s eta 0:00:01
     ------ ------------------------------- 112.6/627.2 kB 1.3 MB/s eta 0:00:01
     ---------- --------------------------- 174.1/627.2 kB 1.2 MB/s eta 0:00:01
     ------------- ------------------------ 225.3/627.2 kB 1.1 MB/s eta 0:00:01
     ----------------- -------------------- 286.7/627.2 kB 1.4 MB/s eta 0:00:01
     ------------------------- ------------ 419.8/627.2 kB 1.5 MB/s eta 0:00:01
     ------------------------------- ------ 522.2/627.2 kB 1.6 MB/s eta 0:00:01
     -------------------------------------  624.6/627.2 kB 1.6 MB/s eta 0:00:01
     -------------------------------------- 627.2/627.2 kB 1.6 MB/s eta 0:00:00



[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
!pip install chardet

WINDOWS-1252


In [31]:
import chardet
with open('C:\\Users\\Riyad\\projects\\medical-assistand-ml\\Doctor_Versus_Disease.csv', 'rb') as f:
    result = chardet.detect(f.read())
    print(result['encoding'])

WINDOWS-1252


In [32]:
import pandas as pd

df_doctor_vs_disease = pd.read_csv(
    'C:\\Users\\Riyad\\projects\\medical-assistand-ml\\Doctor_Versus_Disease.csv',
    encoding='windows-1252'
)

print(df_doctor_vs_disease.shape)
print(df_doctor_vs_disease.columns.tolist())
print(df_doctor_vs_disease.head(10))

(40, 2)
['Drug Reaction', 'Allergist']
      Drug Reaction        Allergist
0           Allergy        Allergist
1     Hypertension      Cardiologist
2      Heart attack     Cardiologist
3         Psoriasis    Dermatologist
4       Chicken pox    Dermatologist
5              Acne    Dermatologist
6          Impetigo    Dermatologist
7  Fungal infection    Dermatologist
8    Hypothyroidism  Endocrinologist
9         Diabetes   Endocrinologist
